# Repeat Assessment Task  

In [54]:
import warnings
warnings.filterwarnings('ignore') # We can suppress the warnings

## Task 
The Fremont Bridge Bicycle Counter began operation in October 2012 and records the number of bikes that cross the bridge using the pedestrian/bicycle pathways. Inductive loops on the east and west pathways count the passing of bicycles regardless of travel direction. The data consists of a date/time field: Date, east pathway count field: Fremont Bridge NB, and west pathway count field: Fremont Bridge SB. The count fields represent the total bicycles detected during the specified one-hour period. Direction of travel is not specified, but in general most traffic in the Fremont Bridge NB field is travelling northbound and most traffic in the Fremont Bridge SB field is travelling southbound.
 Interactive Dashboard for Urban Planners
●	Create an interactive Dashboard aimed at urban planners and community advocates interested in promoting cycling.
●	Summarize key insights from the bicycle counter data (e.g., busiest months, peak cycling hours, seasonal trends).
●	Use clear, engaging visual elements like line plots, heatmaps, time sliders, and filters (for weekday vs weekend traffic, seasonality, etc.).
●	Design the dashboard to be user-friendly, visually appealing, and suitable for stakeholders with varying levels of technical expertise.
●	Explain in your report how your dashboard design and visualisation choices help highlight why time series forecasting and pattern discovery are essential for supporting sustainable urban mobility initiatives.
###

## Step : Importing the required libraries

In [55]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

## Loading the Data

In [56]:
dataset = pd.read_csv('Fremont_Bridge_Bicycle_Counter.csv') #reading the dataset
dataset.head(100)  #displaying the first 100 rows of the dataset, overview of structure and content

,Date,"Fremont Bridge Sidewalks, south of N 34th St Total","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk"
0,10/02/2012 01:00:00 PM,55.0,7.0,48.0
1,10/02/2012 02:00:00 PM,130.0,55.0,75.0
2,10/02/2012 03:00:00 PM,152.0,81.0,71.0
3,10/02/2012 04:00:00 PM,278.0,167.0,111.0
4,10/02/2012 05:00:00 PM,563.0,393.0,170.0
...,...,...,...,...
95,10/06/2012 12:00:00 PM,164.0,73.0,91.0
96,10/06/2012 01:00:00 PM,177.0,82.0,95.0
97,10/06/2012 02:00:00 PM,203.0,114.0,89.0
98,10/06/2012 03:00:00 PM,211.0,96.0,115.0


In [57]:
dataset.info() #checking for data types

<class 'pandas.DataFrame'>
RangeIndex: 110184 entries, 0 to 110183
Data columns (total 4 columns):
 #   Column                                                              Non-Null Count   Dtype  
---  ------                                                              --------------   -----  
 0   Date                                                                110184 non-null  str    
 1   Fremont Bridge Sidewalks, south of N 34th St Total                  110156 non-null  float64
 2   Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk  110156 non-null  float64
 3   Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk  110156 non-null  float64
dtypes: float64(3), str(1)
memory usage: 5.7 MB


In [58]:
dataset.isnull().sum() #checking for missing values

Date                                                                   0
Fremont Bridge Sidewalks, south of N 34th St Total                    28
Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk    28
Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk    28
dtype: int64

In [59]:
#checking summary statistics for numeric columns, 
# to understand distributions and identify potential outliers
dataset.describe()

,"Fremont Bridge Sidewalks, south of N 34th St Total","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk"
count,110156.000000,110156.000000,110156.000000
mean,105.154581,44.831167,60.323414
std,130.736785,58.676107,81.581744
min,0.000000,0.000000,0.000000
25%,13.000000,5.000000,7.000000
50%,59.000000,25.000000,32.000000
75%,144.000000,62.000000,80.000000
max,1097.000000,667.000000,850.000000


In [60]:
dataset.dtypes #checking data types of each column 

Date                                                                      str
Fremont Bridge Sidewalks, south of N 34th St Total                    float64
Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk    float64
Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk    float64
dtype: object

## Handling the Missing Data


In [61]:
# Create a copy of the dataset to preserve the raw data state
df_cleaned = dataset.copy()

# Fill missing numerical values using forward fill (propagates the last valid observation)
df_cleaned.ffill(inplace=True)

# Double-check that no missing values remain
print("Remaining missing values:")
print(df_cleaned.isnull().sum())

Remaining missing values:
Date                                                                  0
Fremont Bridge Sidewalks, south of N 34th St Total                    0
Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk    0
Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk    0
dtype: int64


In [62]:
# Convert Date column to datetime object
# Using errors='coerce' turns invalid formats into NaT safely
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'], errors='coerce')

# Verify the changes
print(df_cleaned['Date'].dtype)

datetime64[us]


In [63]:
# Rename columns for ease of use and professional layout
df_cleaned.rename(columns={
    'Fremont Bridge Sidewalks, south of N 34th St Total': 'Total_Count',
    'Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk': 'West_Sidewalk_SB',
    'Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk': 'East_Sidewalk_NB'
}, inplace=True)

# Review the newly cleaned columns
df_cleaned.head()

,Date,Total_Count,West_Sidewalk_SB,East_Sidewalk_NB
0,2012-10-02 13:00:00,55.0,7.0,48.0
1,2012-10-02 14:00:00,130.0,55.0,75.0
2,2012-10-02 15:00:00,152.0,81.0,71.0
3,2012-10-02 16:00:00,278.0,167.0,111.0
4,2012-10-02 17:00:00,563.0,393.0,170.0


In [64]:
df_cleaned['West_Sidewalk_SB'] = df_cleaned['West_Sidewalk_SB'].fillna(0)
df_cleaned['East_Sidewalk_NB'] = df_cleaned['East_Sidewalk_NB'].fillna(0)

# 2. PLACE THE RE-CREATION LOGIC HERE:
df_cleaned['Total_Count'] = df_cleaned['West_Sidewalk_SB'] + df_cleaned['East_Sidewalk_NB']

print("✅ 'Total_Count' logic successfully synchronized with West and East sidewalk vectors.")

✅ 'Total_Count' logic successfully synchronized with West and East sidewalk vectors.


# Feature Engineering

In [65]:
# Feature Engineering: Extracting core components of time
df_cleaned['Hour'] = df_cleaned['Date'].dt.hour
df_cleaned['Month'] = df_cleaned['Date'].dt.month
df_cleaned['Year'] = df_cleaned['Date'].dt.year

# Day of week mapping (0 = Monday, 6 = Sunday)
df_cleaned['Day_of_Week'] = df_cleaned['Date'].dt.dayofweek
df_cleaned['Day_Name'] = df_cleaned['Date'].dt.day_name()

# Creating a binary marker for Weekday vs. Weekend
df_cleaned['Is_Weekend'] = df_cleaned['Day_of_Week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

# Season Mapping function based on Months
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df_cleaned['Season'] = df_cleaned['Month'].apply(get_season)

# Check out your engineered feature matrix
df_cleaned.head()

,Date,Total_Count,West_Sidewalk_SB,East_Sidewalk_NB,Hour,Month,Year,Day_of_Week,Day_Name,Is_Weekend,Season
0,2012-10-02 13:00:00,55.0,7.0,48.0,13,10,2012,1,Tuesday,Weekday,Autumn
1,2012-10-02 14:00:00,130.0,55.0,75.0,14,10,2012,1,Tuesday,Weekday,Autumn
2,2012-10-02 15:00:00,152.0,81.0,71.0,15,10,2012,1,Tuesday,Weekday,Autumn
3,2012-10-02 16:00:00,278.0,167.0,111.0,16,10,2012,1,Tuesday,Weekday,Autumn
4,2012-10-02 17:00:00,563.0,393.0,170.0,17,10,2012,1,Tuesday,Weekday,Autumn


In [66]:
# Check for exact duplicate rows across all columns
duplicate_count = df_cleaned.duplicated().sum()
print(f"Number of exact duplicate rows: {duplicate_count}")

# Check for duplicate timestamps (which breaks sequential time series continuity)
timestamp_duplicates = df_cleaned.duplicated(subset=['Date']).sum()
print(f"Number of duplicate timestamps: {timestamp_duplicates}")

# If duplicates exist, drop them and keep the first occurrence
if timestamp_duplicates > 0:
    df_cleaned = df_cleaned.drop_duplicates(subset=['Date'], keep='first')
    # Sort chronologically to restore timeline order
    df_cleaned = df_cleaned.sort_values('Date').reset_index(drop=True)
    print("Duplicates dropped and data re-indexed successfully.")

Number of exact duplicate rows: 1416
Number of duplicate timestamps: 1416
Duplicates dropped and data re-indexed successfully.


In [67]:
import plotly.express as px
import plotly.graph_objects as go

In [68]:
# Group data by Hour and Workday status
hourly_trends = df_cleaned.groupby(['Hour', 'Is_Weekend'])['Total_Count'].mean().reset_index()

fig_hourly = px.line(
    hourly_trends, 
    x='Hour', 
    y='Total_Count', 
    color='Is_Weekend',
    title='<b>Peak Cycling Hours: Commuter vs. Recreational Patterns</b>',
    labels={'Total_Count': 'Average Cyclists per Hour', 'Hour': 'Hour of Day (24h)'},
    template='plotly_white',
    markers=True
)

fig_hourly.update_layout(
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    hovermode='x unified'
)
fig_hourly.show()

In [69]:
# Group data by Month and Directional pathways
monthly_trends = df_cleaned.groupby('Month')[['Total_Count', 'West_Sidewalk_SB', 'East_Sidewalk_NB']].mean().reset_index()
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
monthly_trends['Month_Name'] = monthly_trends['Month'].map(month_names)

fig_monthly = go.Figure()
fig_monthly.add_trace(go.Bar(x=monthly_trends['Month_Name'], y=monthly_trends['West_Sidewalk_SB'], name='Southbound (West)', marker_color='#1f77b4'))
fig_monthly.add_trace(go.Bar(x=monthly_trends['Month_Name'], y=monthly_trends['East_Sidewalk_NB'], name='Northbound (East)', marker_color='#ff7f0e'))

fig_monthly.update_layout(
    title='<b>Busiest Months & Pathway Directional Distribution</b>',
    xaxis_title='Month',
    yaxis_title='Average Hourly Bike Count',
    barmode='stack',
    template='plotly_white',
    hovermode='x unified'
)
fig_monthly.show()


In [70]:
# Create a pivot table counting average riders per hour per day of week
heatmap_data = df_cleaned.groupby(['Day_of_Week', 'Hour'])['Total_Count'].mean().reset_index()
# Map index back to string labels for readable visual layout
day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
heatmap_data['Day_Name'] = heatmap_data['Day_of_Week'].map(day_map)

# Pivot for heatmap generation
heatmap_pivot = heatmap_data.pivot(index='Day_Name', columns='Hour', values='Total_Count')
heatmap_pivot = heatmap_pivot.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

fig_heatmap = px.imshow(
    heatmap_pivot,
    labels=dict(x="Hour of the Day", y="Day of the Week", color="Avg Count"),
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    color_continuous_scale='Viridis',
    title='<b>Bridge Congestion Heatmap: Hour vs. Day of Week</b>'
)
fig_heatmap.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig_heatmap.show()

In [71]:
# Resample data to Daily totals to clean up noise for long-term tracking
daily_trends = df_cleaned.set_index('Date').resample('D')[['Total_Count', 'West_Sidewalk_SB', 'East_Sidewalk_NB']].sum().reset_index()

fig_macro = go.Figure()
fig_macro.add_trace(go.Scatter(x=daily_trends['Date'], y=daily_trends['Total_Count'], name='Total Crossings', line=dict(color='#636EFA', width=1.5)))

# Incorporating an interactive Range Slider and micro-selectors
fig_macro.update_layout(
    title='<b>Multi-Year Cyclist Volume Trajectory with Interactive Timeline Selector</b>',
    xaxis_title='Timeline',
    yaxis_title='Total Daily Bike Volume',
    template='plotly_white',
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    )
)
fig_macro.show()

In [72]:
# Seasonal distribution boxplot
fig_season = px.box(
    df_cleaned, 
    x='Season', 
    y='Total_Count', 
    color='Season',
    category_orders={'Season': ['Spring', 'Summer', 'Autumn', 'Winter']},
    title='<b>Bicycle Volume Distribution Across Seasons</b>',
    labels={'Total_Count': 'Hourly Cyclist Volume', 'Season': 'Season'},
    template='plotly_white'
)

fig_season.show()

In [73]:
# Filter for Weekdays and aggregate by Hour for Northbound vs Southbound
weekday_df = df_cleaned[df_cleaned['Is_Weekend'] == 'Weekday']
directional_hourly = weekday_df.groupby('Hour')[['West_Sidewalk_SB', 'East_Sidewalk_NB']].mean().reset_index()

fig_direction = go.Figure()

fig_direction.add_trace(go.Scatter(
    x=directional_hourly['Hour'], 
    y=directional_hourly['West_Sidewalk_SB'],
    mode='lines+markers',
    name='Southbound (West Pathway)',
    line=dict(color='#1f77b4', width=2)
))

fig_direction.add_trace(go.Scatter(
    x=directional_hourly['Hour'], 
    y=directional_hourly['East_Sidewalk_NB'],
    mode='lines+markers',
    name='Northbound (East Pathway)',
    line=dict(color='#ff7f0e', width=2)
))

fig_direction.update_layout(
    title='<b>Weekday Directional Flow: AM Southbound vs. PM Northbound Commute</b>',
    xaxis=dict(title='Hour of Day (24h)', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title='Average Hourly Count'),
    template='plotly_white',
    hovermode='x unified'
)

fig_direction.show()

In [74]:
# Yearly aggregated cycling volume
yearly_df = df_cleaned.groupby('Year')['Total_Count'].sum().reset_index()

fig_yearly = px.bar(
    yearly_df, 
    x='Year', 
    y='Total_Count',
    text_auto='.2s',
    title='<b>Annual Total Bicycle Commute Growth</b>',
    labels={'Total_Count': 'Total Annual Cyclists', 'Year': 'Year'},
    template='plotly_white',
    color='Total_Count',
    color_continuous_scale='Viridis'
)

fig_yearly.update_layout(coloraxis_showscale=False)
fig_yearly.show()

## Dashboard narrative and design rationale
This dashboard is designed for urban planners and community advocates who need a quick, intuitive view of cycling demand. The long-range trend line highlights growth and change over time, the hourly profile shows when cycling is strongest, the heatmap identifies congestion patterns by day and hour, and the seasonal bars reveal when infrastructure support is most needed. The interactive filters make it easier to compare weekday and weekend behavior, explore seasonal patterns, and justify why time-series forecasting and pattern discovery are valuable for sustainable urban mobility planning.

## Key insights from the analysis
- The busiest cycling periods are concentrated in warmer months, showing strong seasonal demand and confirming that weather and daylight strongly influence ridership.
- Weekday patterns show stronger commuter peaks during typical morning and evening travel hours, while weekend traffic is more evenly spread across the day.
- The heatmap highlights concentrated movement at specific hours and weekdays, which is useful for planning safe cycling infrastructure and traffic management.
- The long-range trend view shows that cycling activity is not random; it follows recurring temporal patterns that can be forecast and monitored over time.

These insights support the case for time-series forecasting and pattern discovery as practical tools for sustainable urban mobility planning.

# required install library

In [75]:
#pip install streamlit plotly

In [76]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Ensure layout extends fully across browser screen spaces
st.set_page_config(page_title="Fremont Active Mobility Dashboard", layout="wide")

st.title(" FREMONT BRIDGE ACTIVE MOBILITY ANALYTICS")
st.markdown("### Executive Planning Interactive Control Panel")

# dataloader
@st.cache_data
def load_dashboard_data():
    df = pd.read_csv("fremont_cleaned.csv")
    df['Date'] = pd.to_datetime(df['Date'])
    return df

try:
    df_cleaned = load_dashboard_data()
    
 #sidebar controlls
    st.sidebar.header("Interactive Control Board")
    
    # timeline Window Filter Slider Matrix

    min_date = df_cleaned['Date'].min().date()
    max_date = df_cleaned['Date'].max().date()
    selected_dates = st.sidebar.date_input("Filter Data by Date Range", [min_date, max_date])

    
    
    # categorical Profile Context Dropdown Selector

    traffic_profile = st.sidebar.selectbox(
        "Filter Data by Day Category Profile", 
        ["Show All Traffic Volume", "Weekday Commuters Only", "Weekend Leisure Only"]
    )
    
    # baseline layout filtering conditions
    if len(selected_dates) == 2:
        start_date, end_date = selected_dates
        filtered_df = df_cleaned[(df_cleaned['Date'].dt.date >= start_date) & (df_cleaned['Date'].dt.date <= end_date)]
    else:
        filtered_df = df_cleaned.copy()
        
    if traffic_profile == "Weekday Commuters Only":
        filtered_df = filtered_df[filtered_df['Is_Weekend'] == 'Weekday']
    elif traffic_profile == "Weekend Leisure Only":
        filtered_df = filtered_df[filtered_df['Is_Weekend'] == 'Weekend']

# executive KPI Cards - first row of dashboard
    st.subheader("1. Long-Term Cycling Growth Trends (2012 - Present)")
    daily_trends = filtered_df.set_index('Date').resample('D')[['Total_Count']].sum().reset_index()
    
    fig_macro = px.line(daily_trends, x='Date', y='Total_Count', labels={'Total_Count': 'Daily Totals'})
    fig_macro.update_layout(template='plotly_white', height=400)
    st.plotly_chart(fig_macro, use_container_width=True)
    
    st.markdown("<br>", unsafe_allow_html=True)

    # peak hours commuter analysis and infrastructure grid traffic
    col1, col2 = st.columns(2)
    
    with col1:
        st.subheader("2. Busiest Hours of the Day: Weekdays vs. Weekends")
        hourly_trends = filtered_df.groupby(['Hour', 'Is_Weekend'])['Total_Count'].mean().reset_index()
        fig_hourly = px.line(hourly_trends, x='Hour', y='Total_Count', color='Is_Weekend', markers=True)
        fig_hourly.update_layout(template='plotly_white', xaxis=dict(tickmode='linear', dtick=2), height=400)
        st.plotly_chart(fig_hourly, use_container_width=True)
        
    with col2:
   st.subheader("3. Hourly Traffic Patterns by Day of the Week")
        
        if not filtered_df.empty:
            # 1. Aggregate and map days
            day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
            heatmap_data = filtered_df.groupby(['Day_of_Week', 'Hour'])['Total_Count'].mean().reset_index()
            heatmap_data['Day_Name'] = heatmap_data['Day_of_Week'].map(day_map)
            
            # 2. Pivot and reindex to guarantee chronological day ordering
            heatmap_pivot = heatmap_data.pivot(index='Day_Name', columns='Hour', values='Total_Count').reindex(
                ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
            )
            
            # 3. Generate heatmap with explicit axis titles and clean formatting
            fig_heat = px.imshow(
                heatmap_pivot, 
                color_continuous_scale='Viridis',
                labels=dict(x="Hour of Day (24h)", y="Day of Week", color="Avg Count"),
                text_auto='.0f'  # Optional: Adds count values directly on the blocks if zoomed in
            )
            
            fig_heat.update_layout(
                xaxis=dict(tickmode='linear', dtick=2), 
                height=400,
                margin=dict(l=20, r=20, t=30, b=20)
            )
            
            st.plotly_chart(fig_heat, use_container_width=True)
        else:
            st.warning("⚠️ No data available for the selected filters to generate the heatmap.")

    st.markdown("<br><hr><br>", unsafe_allow_html=True)

#seasonal crossings variation analysis and strategic individual project management
    col3, col4 = st.columns(2)
    
    with col3:
        st.subheader("4. Seasonal Crossings Variation Analysis")
        month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
        monthly_trends = filtered_df.groupby('Month')[['West_Sidewalk_SB', 'East_Sidewalk_NB']].mean().reset_index()
        monthly_trends['Month_Name'] = monthly_trends['Month'].map(month_names)
        
        fig_seasonal = go.Figure()
        fig_seasonal.add_trace(go.Bar(x=monthly_trends['Month_Name'], y=monthly_trends['West_Sidewalk_SB'], name='Southbound Pathway', marker_color='#2E4053'))
        fig_seasonal.add_trace(go.Bar(x=monthly_trends['Month_Name'], y=monthly_trends['East_Sidewalk_NB'], name='Northbound Pathway', marker_color='#5DADE2'))
        fig_seasonal.update_layout(barmode='stack', template='plotly_white', height=400)
        st.plotly_chart(fig_seasonal, use_container_width=True)
        
    with col4:
        st.subheader("5. Strategic Individual Project Management Effort Matrix")
        effort_df = pd.DataFrame({
            'Task Phase Area': ['Data Wrangling', 'Feature Extraction', 'Dashboard Formulation', 'Technical Report Writeup'],
            'Hours Accounted Log': [6, 4, 12, 8]
        })
        accessibility_hues = ['#1B4F72', '#0E6251', '#7D6608', '#4A235A']
        fig_effort = px.bar(
            effort_df, x='Task Phase Area', y='Hours Accounted Log', color='Task Phase Area',
            color_discrete_sequence=accessibility_hues, text='Hours Accounted Log'
        )
        fig_effort.update_layout(template='plotly_white', showlegend=False, height=400)
        st.plotly_chart(fig_effort, use_container_width=True)

except FileNotFoundError:
    st.error("Error configuration context missing pipeline asset baseline. Please initialize database extraction step cell first.")

Overwriting dashboard.py


In [77]:
import numpy as np
import pandas as pd

# Load fresh data to guarantee the baseline columns exist
df_cleaned = pd.read_csv("fremont_cleaned.csv")

# 1. Force the binary column to create itself
df_cleaned['Rush_Hour_Flag'] = np.where(
    (df_cleaned['Is_Weekend'] == 'Weekday') & 
    (((df_cleaned['Hour'] >= 7) & (df_cleaned['Hour'] <= 9)) | 
     ((df_cleaned['Hour'] >= 16) & (df_cleaned['Hour'] <= 18))),
    'Rush Hour', 'Off-Peak'
)

# 2. Force the behavior buckets column
def assign_time_bucket(hour):
    if 5 <= hour <= 11: return 'Morning'
    elif 12 <= hour <= 16: return 'Afternoon'
    elif 17 <= hour <= 21: return 'Evening'
    else: return 'Night'
df_cleaned['Time_of_Day_Bucket'] = df_cleaned['Hour'].apply(assign_time_bucket)

# 3. Save it to overwrite your disk asset completely
df_cleaned.to_csv("fremont_cleaned.csv", index=False)
print("✅ Column array successfully injected! Your data asset file now contains 'Rush_Hour_Flag'.")

✅ Column array successfully injected! Your data asset file now contains 'Rush_Hour_Flag'.


In [86]:
import os
import sys
import time
import socket
import subprocess
import numpy as np
import pandas as pd

print("⏳ Initializing Data Processing Pipeline...")

try:
    df_temp = pd.read_csv("fremont_cleaned.csv")
except FileNotFoundError:
    print("❌ Error: 'fremont_cleaned.csv' not found. Ensure the dataset exists in your directory.")
    sys.exit(1)

# Datetime Parsing
df_temp['Date'] = pd.to_datetime(df_temp['Date'], errors='coerce')

# Missing Value Imputation
df_temp['West_Sidewalk_SB'] = df_temp['West_Sidewalk_SB'].fillna(0)
df_temp['East_Sidewalk_NB'] = df_temp['East_Sidewalk_NB'].fillna(0)

# Mathematical Consistency Enforcement
df_temp['Total_Count'] = df_temp['West_Sidewalk_SB'] + df_temp['East_Sidewalk_NB']

# Hour Extraction & Fallback Synchronization
if 'Hour' in df_temp.columns:
    df_temp['Hour'] = df_temp['Date'].dt.hour.fillna(df_temp['Hour']).astype(int)
else:
    df_temp['Hour'] = df_temp['Date'].dt.hour.astype(int)

# Deduplication & Sorting
df_temp = df_temp.drop_duplicates(subset=['Date']).sort_values('Date').reset_index(drop=True)

# MinMax Normalization
c_min, c_max = df_temp['Total_Count'].min(), df_temp['Total_Count'].max()
df_temp['Total_Count_MinMax'] = (df_temp['Total_Count'] - c_min) / (c_max - c_min) if (c_max - c_min) > 0 else 0

# Standard Z-Score Normalization
c_mean, c_std = df_temp['Total_Count'].mean(), df_temp['Total_Count'].std()
df_temp['Total_Count_ZScore'] = (df_temp['Total_Count'] - c_mean) / c_std if c_std > 0 else 0

# Rush Hour Flag
df_temp['Rush_Hour_Flag'] = np.where(
    (df_temp['Is_Weekend'] == 'Weekday') & 
    (((df_temp['Hour'] >= 7) & (df_temp['Hour'] <= 9)) | 
     ((df_temp['Hour'] >= 16) & (df_temp['Hour'] <= 18))),
    'Rush Hour', 'Off-Peak'
)

# Time-of-Day Buckets
def assign_time_bucket(hour):
    if 5 <= hour <= 11: 
        return 'Morning'
    elif 12 <= hour <= 16: 
        return 'Afternoon'
    elif 17 <= hour <= 21: 
        return 'Evening'
    else: 
        return 'Night'

df_temp['Time_of_Day_Bucket'] = df_temp['Hour'].apply(assign_time_bucket)

# Volatility Index
volatility_lookup = df_temp.groupby('Day_Name')['Total_Count'].std().to_dict()
df_temp['Day_Volatility_Index'] = df_temp['Day_Name'].map(volatility_lookup)

# Seasonal Impact Multiplier Index
global_median = df_temp['Total_Count'].median() if df_temp['Total_Count'].median() > 0 else 1
seasonal_avg = df_temp.groupby('Season')['Total_Count'].mean()
seasonal_index_lookup = (seasonal_avg / global_median).to_dict()
df_temp['Seasonal_Impact_Multiplier'] = df_temp['Season'].map(seasonal_index_lookup)

# Save Processed Dataset
df_temp.to_csv("fremont_cleaned.csv", index=False)
print("✅ Data processing clean; asset matrix written safely to 'fremont_cleaned.csv'")


dashboard_code = """import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Page Layout Config
st.set_page_config(page_title="Fremont Ultimate Mobility Framework", layout="wide")

# Custom Styling
st.markdown(\"\"\"
    <style>
    .main { background-color: #F8F9FA; }
    h1 { color: #0F2C59; font-family: 'Helvetica Neue', Arial, sans-serif; font-weight: 800; }
    h2, h3 { color: #1E2022; font-family: 'Helvetica Neue', Arial, sans-serif; }
    div[data-testid="stMetricValue"] { font-size: 32px; color: #0284C7; font-weight: bold; }
    </style>
\"\"\", unsafe_allow_html=True)

st.title("FREMONT BRIDGE ACTIVE MOBILITY ANALYTICS")
st.markdown("### Executive Planning & Data-Driven Infrastructure Control Deck")
st.markdown("---")

@st.cache_data
def load_dashboard_data():
    df = pd.read_csv("fremont_cleaned.csv")
    df['Date'] = pd.to_datetime(df['Date'])
    
    if 'Rush_Hour_Flag' not in df.columns:
        df['Rush_Hour_Flag'] = np.where(
            (df['Is_Weekend'] == 'Weekday') & 
            (((df['Hour'] >= 7) & (df['Hour'] <= 9)) | 
             ((df['Hour'] >= 16) & (df['Hour'] <= 18))),
            'Rush Hour', 'Off-Peak'
        )
    if 'Time_of_Day_Bucket' not in df.columns:
        def assign_time_bucket(hour):
            if 5 <= hour <= 11: return 'Morning'
            elif 12 <= hour <= 16: return 'Afternoon'
            elif 17 <= hour <= 21: return 'Evening'
            else: return 'Night'
        df['Time_of_Day_Bucket'] = df['Hour'].apply(assign_time_bucket)
    return df

try:
    df_cleaned = load_dashboard_data()
    
    # Sidebar Filters
    st.sidebar.header("🕹️ Controls & Filters")
    min_date = df_cleaned['Date'].min().date()
    max_date = df_cleaned['Date'].max().date()
    
    selected_dates = st.sidebar.date_input(
        "Date Constraints Range", 
        [min_date, max_date],
        format="DD/MM/YYYY"
    )
    
    traffic_profile = st.sidebar.selectbox(
        "Traffic Profiler Filter Profile", 
        ["Show All Traffic Volume", "Weekday Commuters Only", "Weekend Leisure Only"]
    )
    
    if len(selected_dates) == 2:
        start_date, end_date = selected_dates
        filtered_df = df_cleaned[(df_cleaned['Date'].dt.date >= start_date) & (df_cleaned['Date'].dt.date <= end_date)]
        start_str = start_date.strftime('%d/%m/%Y')
        end_str = end_date.strftime('%d/%m/%Y')
        st.info(f"📅 Active Analysis Window Filtered From: **{start_str}** to **{end_str}**")
    else:
        filtered_df = df_cleaned.copy()
        st.info(f"📅 Active Analysis Window Filtered From: **{min_date.strftime('%d/%m/%Y')}** to **{max_date.strftime('%d/%m/%Y')}**")
        
    if traffic_profile == "Weekday Commuters Only":
        filtered_df = filtered_df[filtered_df['Is_Weekend'] == 'Weekday']
    elif traffic_profile == "Weekend Leisure Only":
        filtered_df = filtered_df[filtered_df['Is_Weekend'] == 'Weekend']

    # KPI Summary Cards
    kpi1, kpi2, kpi3 = st.columns(3)
    with kpi1:
        st.metric("Total Documented Crossings", f"{int(filtered_df['Total_Count'].sum()):,}")
    with kpi2:
        st.metric("Average Hourly Passing Rate", f"{round(filtered_df['Total_Count'].mean(), 1)} riders/hr")
    with kpi3:
        st.metric("Busiest Recorded Peak Hour Volume", f"{int(filtered_df['Total_Count'].max()):,} riders")
        
    st.markdown("<br>", unsafe_allow_html=True)

    # Section A: Baseline Visualizations
    st.markdown("## Core Dataset Baseline Performance Metrics")
    
    # 1. Timeline Chart
    st.subheader("1. Long-Term Cycling Growth Trends")
    daily_trends = filtered_df.set_index('Date').resample('D')[['Total_Count']].sum().reset_index()
    fig_macro = px.line(daily_trends, x='Date', y='Total_Count', color_discrete_sequence=['#1E3A8A'])
    fig_macro.update_layout(
        template='plotly_white', 
        height=420, 
        margin=dict(l=20, r=20, t=10, b=20),
        xaxis=dict(rangeslider=dict(visible=True), type="date")
    )
    st.plotly_chart(fig_macro, use_container_width=True)

    # 2 & 3. Hourly & Heatmap Charts
    col_orig1, col_orig2 = st.columns(2)
    with col_orig1:
        st.subheader("2. Busiest Hours: Weekdays vs. Weekends")
        hourly_trends = filtered_df.groupby(['Hour', 'Is_Weekend'])['Total_Count'].mean().reset_index()
        fig_hourly = px.line(hourly_trends, x='Hour', y='Total_Count', color='Is_Weekend', markers=True,
                             color_discrete_sequence=['#10B981', '#F59E0B'])
        fig_hourly.update_layout(template='plotly_white', xaxis=dict(tickmode='linear', dtick=2), height=360)
        st.plotly_chart(fig_hourly, use_container_width=True)
        
    with col_orig2:
        st.subheader("3. Traffic Intensity Heatmap")
        day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
        heatmap_data = filtered_df.groupby(['Day_of_Week', 'Hour'])['Total_Count'].mean().reset_index()
        heatmap_data['Day_Name'] = heatmap_data['Day_of_Week'].map(day_map)
        heatmap_pivot = heatmap_data.pivot(index='Day_Name', columns='Hour', values='Total_Count').reindex(
            ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        )
        fig_heat = px.imshow(heatmap_pivot, color_continuous_scale='Viridis')
        fig_heat.update_layout(xaxis=dict(tickmode='linear', dtick=2), height=360)
        st.plotly_chart(fig_heat, use_container_width=True)

    # Section B: Targeted Urban Planning Visualizations
    st.markdown("<br><hr><br>", unsafe_allow_html=True)
    st.markdown("## Infrastructure & Policy Planning Analytics")

    # Directional Commute Split & Seasonal Box Plot
    col_plan1, col_plan2 = st.columns(2)
    
    with col_plan1:
        st.subheader("4. Directional Commute Split (Asymmetric Hourly Usage)")
        dir_trends = filtered_df.groupby('Hour')[['West_Sidewalk_SB', 'East_Sidewalk_NB']].mean().reset_index()
        fig_dir = go.Figure()
        fig_dir.add_trace(go.Scatter(x=dir_trends['Hour'], y=dir_trends['West_Sidewalk_SB'], mode='lines+markers', name='Southbound (West)', line=dict(color='#2563EB')))
        fig_dir.add_trace(go.Scatter(x=dir_trends['Hour'], y=dir_trends['East_Sidewalk_NB'], mode='lines+markers', name='Northbound (East)', line=dict(color='#D97706')))
        fig_dir.update_layout(
            template='plotly_white',
            height=360,
            xaxis=dict(tickmode='linear', dtick=2, title='Hour of Day'),
            yaxis=dict(title='Avg Hourly Cyclist Volume'),
            hovermode='x unified'
        )
        st.plotly_chart(fig_dir, use_container_width=True)
        

    with col_plan2:
        st.subheader("5. Seasonal Traffic Distribution (Weather Drop-offs)")
        fig_box = px.box(
            filtered_df, 
            x='Season', 
            y='Total_Count', 
            color='Season',
            category_orders={'Season': ['Spring', 'Summer', 'Autumn', 'Winter']},
            color_discrete_map={'Spring': '#10B981', 'Summer': '#F59E0B', 'Autumn': '#D97706', 'Winter': '#3B82F6'}
        )
        fig_box.update_layout(
            template='plotly_white',
            height=360,
            showlegend=False,
            yaxis=dict(title='Hourly Rider Count')
        )
        st.plotly_chart(fig_box, use_container_width=True)
       

    # Year-over-Year Growth & Rush Hour Donut
    st.markdown("<br>", unsafe_allow_html=True)
    col_plan3, col_plan4 = st.columns(2)

    with col_plan3:
        st.subheader("6. Year-over-Year Overall Traffic Growth")
        yoy_df = filtered_df.copy()
        yoy_df['Year'] = yoy_df['Date'].dt.year
        yearly_summary = yoy_df.groupby('Year')['Total_Count'].sum().reset_index()
        
        fig_yoy_growth = px.bar(
            yearly_summary, 
            x='Year', 
            y='Total_Count',
            text_auto='.2s',
            color_discrete_sequence=['#0EA5E9']
        )
        fig_yoy_growth.update_layout(
            template='plotly_white',
            height=360,
            xaxis=dict(tickmode='linear', dtick=1),
            yaxis=dict(title='Total Annual Cyclist Volume')
        )
        st.plotly_chart(fig_yoy_growth, use_container_width=True)
        

    with col_plan4:
        st.subheader("7. Rush Hour vs. Off-Peak Proportion")
        rush_data = filtered_df.groupby('Rush_Hour_Flag')['Total_Count'].sum().reset_index()
        fig_donut = px.pie(
            rush_data, 
            values='Total_Count', 
            names='Rush_Hour_Flag', 
            hole=0.5,
            color_discrete_sequence=['#0284C7', '#CBD5E1']
        )
        fig_donut.update_layout(height=360, margin=dict(l=20, r=20, t=20, b=20))
        st.plotly_chart(fig_donut, use_container_width=True)


    # Section C: Engineered Volatility & Project Effort Features
    st.markdown("<br><hr><br>", unsafe_allow_html=True)
    st.markdown("## Engineered Volatility & Project Operations")

    col_adv1, col_adv2 = st.columns(2)
    with col_adv1:
        st.subheader("8. Daily Traffic Volatility Index (Standard Deviation)")
        vol_data = filtered_df.groupby('Day_Name')['Total_Count'].std().reset_index().rename(columns={'Total_Count': 'Volatility_SD'})
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        vol_data['Day_Name'] = pd.Categorical(vol_data['Day_Name'], categories=day_order, ordered=True)
        vol_data = vol_data.sort_values('Day_Name')
        fig_vol = px.line(vol_data, x='Day_Name', y='Volatility_SD', markers=True, color_discrete_sequence=['#EF4444'])
        fig_vol.update_layout(height=350, template='plotly_white', yaxis_title="Standard Deviation (Sigma)")
        st.plotly_chart(fig_vol, use_container_width=True)
        
    with col_adv2:
        st.subheader("9. Project Management Effort Allocation Matrix")
        effort_df = pd.DataFrame({
            'Task Phase Area': ['Data Wrangling', 'Feature Extraction', 'Dashboard Formulation', 'Technical Report Writeup'],
            'Hours Accounted Log': [6, 4, 12, 8]
        })
        fig_effort = px.bar(effort_df, x='Task Phase Area', y='Hours Accounted Log', color='Task Phase Area',
                            color_discrete_sequence=['#1E3A8A', '#0D9488', '#B45309', '#6B21A8'], text='Hours Accounted Log')
        fig_effort.update_layout(template='plotly_white', showlegend=False, height=350)
        st.plotly_chart(fig_effort, use_container_width=True)

except Exception as e:
    st.error(f"Error loading dashboard application components: {e}")
"""

with open("dashboard.py", "w", encoding="utf-8") as f:
    f.write(dashboard_code)

print("👍 Dashboard file generated successfully as 'dashboard.py'.")

# ==============================================================================
# STEP 3: AUTOMATED PROCESS RUNNER & SERVER INITIALIZATION
# ==============================================================================
def automated_process_runner(port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        is_port_in_use = s.connect_ex(('localhost', port)) == 0

    if is_port_in_use:
        print(f"🔗 Dashboard server is already active: http://localhost:{port}")
    else:
        print(f"🚀 Initializing Streamlit dashboard server on port {port}...")
        cmd = [sys.executable, '-m', 'streamlit', 'run', 'dashboard.py', '--server.port', str(port)]
        
        if os.name == 'nt':
            subprocess.Popen(cmd, creationflags=0x00000008)  # DETACHED_PROCESS for Windows
        else:
            subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=os.setpgrp)
            
        time.sleep(2.5)
        print(f"🌐 Deployment Success! Open dashboard at: http://localhost:{port}")

if __name__ == "__main__":
    automated_process_runner()

⏳ Initializing Data Processing Pipeline...
✅ Data processing clean; asset matrix written safely to 'fremont_cleaned.csv'
👍 Dashboard file generated successfully as 'dashboard.py'.
🔗 Dashboard server is already active: http://localhost:8501
